# Assignment 3 
#### Name - Nitish Kumar
#### Roll No - 231110033
#### Dept. - M.Tech(C.S.E)
## For the Model NLLB

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import random


tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


# Importing English File

In [3]:
with open("/kaggle/input/dataset/test.en", 'r') as file:
    
    sen_in_english = file.readlines()
    
sen_in_english = [sentence.strip() for sentence in sen_in_english]


# Importing Hindi File

In [4]:
with open("/kaggle/input/dataset/test.hi", 'r') as file:
    
    sen_in_hindi = file.readlines()
    
sen_in_hindi = [sentence.strip() for sentence in sen_in_hindi]


# Importing Gujrati File

In [5]:
with open("/kaggle/input/dataset/test.gu", 'r') as file:
    
    sen_in_gujarati = file.readlines()
    
sen_in_gujarati = [sentence.strip() for sentence in sen_in_gujarati]


### Randomly Selecting a Thousand Sentences from Hindi, English, and Gujarati Data and using seed to take care of same random sentences for all three languages 

In [6]:
random.seed(1)
index_of_sampled_sentences = random.sample(range(len(sen_in_english)), 1000)
english_sentences_sampled = [sen_in_english[i] for i in index_of_sampled_sentences]
hindi_sentences_sampled = [sen_in_hindi[i] for i in index_of_sampled_sentences]
gujarati_sentences_sampled = [sen_in_gujarati[i] for i in index_of_sampled_sentences]

# Translating English to Hindi

In [8]:
english_to_hindi_translation=[]
for i in range(0,20):
    inputs = tokenizer(english_sentences_sampled[i*50:(i*50)+50], return_tensors="pt", padding = "longest")
    translated_tokens = model.generate(
        **inputs, forced_bos_token_id=tokenizer.lang_code_to_id["hin_Deva"], max_length= 32
    )
    english_to_hindi_translation.extend(tokenizer.batch_decode(translated_tokens, skip_special_tokens=True))

#### For rerun I'm saving the translated file locally 

In [9]:
with open("/kaggle/working/Eng_to_Hindi.txt", "w",encoding="utf-8") as file:
    # Write each sentence to the file followed by a newline character
    for sentence in english_to_hindi_translation:
        file.write(sentence + "\n")

In [10]:
# english_to_hindi_translation

# Translating Hindi to English

In [12]:
hindi_to_english_translation=[]
for i in range(0,20):
    inputs = tokenizer(hindi_sentences_sampled[i*50:(i*50)+50], return_tensors="pt", padding = "longest")

    translated_tokens = model.generate(
        **inputs, forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"], max_length= 32
    )
    hindi_to_english_translation.extend(tokenizer.batch_decode(translated_tokens, skip_special_tokens=True))

#### For rerun I'm saving the translated file locally 

In [13]:
with open("/kaggle/working/Hindi_to_Eng.txt", "w",encoding="utf-8") as file:
    # Write each sentence to the file followed by a newline character
    for sentence in hindi_to_english_translation:
        file.write(sentence + "\n")

In [ ]:
# hindi_to_english_translation

# Translating Hindi to Gujrati

In [14]:
translated_hin_to_guj = []
for i in range(0,20):

    inputs = tokenizer(hindi_sentences_sampled[i*50:(i*50)+50], return_tensors="pt", padding = "longest")

    translated_tokens = model.generate(
        **inputs, forced_bos_token_id=tokenizer.lang_code_to_id["guj_Gujr"], max_length= 32
    )
    translated_hin_to_guj.extend(tokenizer.batch_decode(translated_tokens, skip_special_tokens=True))

#### For rerun I'm saving the translated file locally 

In [15]:
with open("/kaggle/working//Hindi_to_Guj.txt", "w",encoding="utf-8") as file:
    # Write each sentence to the file followed by a newline character
    for sentence in translated_hin_to_guj:
        file.write(sentence + "\n")

In [ ]:
# translated_hin_to_guj

# Translating Gujrati to Hindi

In [16]:
gujarati_to_hindi_translation= []
for i in range(0,20):

    inputs = tokenizer(gujarati_sentences_sampled[i*50:(i*50)+50], return_tensors="pt", padding = "longest")

    translated_tokens = model.generate(
        **inputs, forced_bos_token_id=tokenizer.lang_code_to_id["hin_Deva"], max_length= 32
    )
    gujarati_to_hindi_translation.extend(tokenizer.batch_decode(translated_tokens, skip_special_tokens=True))

#### For rerun I'm saving the translated file locally 

In [17]:
with open("/kaggle/working/Guj_to_Hindi.txt", "w",encoding="utf-8") as file:
    # Write each sentence to the file followed by a newline character
    for sentence in gujarati_to_hindi_translation:
        file.write(sentence + "\n")

In [ ]:
# gujarati_to_hindi_translation

# Evaluation of models

### Calculation of BLEU Score

In [18]:
from nltk.translate.bleu_score import corpus_bleu

# Function to calculate BLEU scores
def calculate_bleu_scores(references, translations):
    # Wrapping each reference sentence in a list because corpus_bleu expects a list of lists
    formatted_references = [[ref] for ref in references]
    bleu_score = corpus_bleu(formatted_references, translations)
    return bleu_score

def print_bleu_scores(model_name, references, translations):
    bleu_score = calculate_bleu_scores(references, translations)
    print(f"\nBLEU Score for {model_name}: {bleu_score:.4f}")

# Calculate and print BLEU scores for each translation model
print_bleu_scores("English to Hindi", hindi_sentences_sampled, english_to_hindi_translation)
print_bleu_scores("Hindi to English", english_sentences_sampled, hindi_to_english_translation)
print_bleu_scores("Hindi to Gujarati", gujarati_sentences_sampled, translated_hin_to_guj)
print_bleu_scores("Gujarati to Hindi", hindi_sentences_sampled, gujarati_to_hindi_translation)



BLEU Score for English to Hindi: 0.6369

BLEU Score for Hindi to English: 0.6862

BLEU Score for Hindi to Gujarati: 0.5836

BLEU Score for Gujarati to Hindi: 0.6387


In [24]:
from nltk.translate.bleu_score import sentence_bleu

# Function to calculate sentence-level BLEU scores and return the average
def calculate_average_sentence_bleu(references, translations):
    score=0
    for ref, trans in zip(references, translations):
        score += sentence_bleu([ref], trans)
    # Calculate the average of the sentence BLEU scores
    average_score = score / 1000 #if scores else 0
    return average_score

def print_average_sentence_bleu(model_name, references, translations):
    average_score = calculate_average_sentence_bleu(references, translations)
    print(f"\nAverage Sentence-level BLEU Score for {model_name}: {average_score:.4f}")

# Printing the Average Sentence-level BLEU Score for each translation
print_average_sentence_bleu("English to Hindi", hindi_sentences_sampled, english_to_hindi_translation)
print_average_sentence_bleu("Hindi to English", english_sentences_sampled, hindi_to_english_translation)
print_average_sentence_bleu("Hindi to Gujarati", gujarati_sentences_sampled, translated_hin_to_guj)
print_average_sentence_bleu("Gujarati to Hindi", hindi_sentences_sampled, gujarati_to_hindi_translation)


Average Sentence-level BLEU Score for English to Hindi: 0.6164

Average Sentence-level BLEU Score for Hindi to English: 0.6502

Average Sentence-level BLEU Score for Hindi to Gujarati: 0.5768

Average Sentence-level BLEU Score for Gujarati to Hindi: 0.6191


### Calculation of ROGUE Score

In [22]:
pip install rouge

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [23]:
import numpy as np
from rouge import Rouge

# Initialize the Rouge scorer
rouge = Rouge()

def average_rouge_scores(translated_texts, reference_texts):
    """

    Args:
        translated_texts (list of str): The translated sentences.
        reference_texts (list of str): The ground truth reference sentences.

    Returns:
        dict: A dictionary containing averaged scores for ROUGE-1, ROUGE-2, and ROUGE-L.
    """
    scores = rouge.get_scores(translated_texts, reference_texts, avg=True)  # Use avg=True to get the average score
    return scores

def print_rouge_scores(model_name, scores):
    """

    Args:
        model_name (str): Description of the model translation direction.
        scores (dict): A dictionary containing averaged ROUGE scores.
    """
    print(f"\nROUGE Scores for {model_name} Translation:")
    print(f"{'Metric':<8} | {'Precision':<10} | {'Recall':<10} | {'F1-score':<10}")
    print('-' * 42)
    for key in ['rouge-1', 'rouge-2', 'rouge-l']:
        metric_name = key.upper().replace('ROUGE-', 'ROUGE ')
        precision = scores[key]['p']
        recall = scores[key]['r']
        f1_score = scores[key]['f']
        print(f"{metric_name:<8} | {precision:<10.4f} | {recall:<10.4f} | {f1_score:<10.4f}")

# Printing ROUGE Scores for each translation model
rouge_score_eng_to_hin = average_rouge_scores(english_to_hindi_translation, hindi_sentences_sampled)
print_rouge_scores("English to Hindi", rouge_score_eng_to_hin)

rouge_score_hin_to_eng = average_rouge_scores(hindi_to_english_translation, english_sentences_sampled)
print_rouge_scores("Hindi to English", rouge_score_hin_to_eng)

rouge_score_hin_to_guj = average_rouge_scores(translated_hin_to_guj, gujarati_sentences_sampled)
print_rouge_scores("Hindi to Gujarati", rouge_score_hin_to_guj)

rouge_score_guj_to_hin = average_rouge_scores(gujarati_to_hindi_translation, hindi_sentences_sampled)
print_rouge_scores("Gujarati to Hindi", rouge_score_guj_to_hin)


ROUGE Scores for English to Hindi Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  | 0.6063     | 0.5743     | 0.5854    
ROUGE 2  | 0.3586     | 0.3393     | 0.3458    
ROUGE L  | 0.5657     | 0.5357     | 0.5461    

ROUGE Scores for Hindi to English Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  | 0.6237     | 0.6135     | 0.6139    
ROUGE 2  | 0.3899     | 0.3873     | 0.3854    
ROUGE L  | 0.5890     | 0.5799     | 0.5801    

ROUGE Scores for Hindi to Gujarati Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  | 0.5254     | 0.4848     | 0.4989    
ROUGE 2  | 0.2774     | 0.2548     | 0.2625    
ROUGE L  | 0.5020     | 0.4634     | 0.4768    

ROUGE Scores for Gujarati to Hindi Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  |